In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)

# 1. Load the Financial PhraseBank Dataset (Secure Parquet Branch)
print("Loading dataset...")
dataset = load_dataset(
    "takala/financial_phrasebank",
    "sentences_75agree",
    revision="refs/pr/10"
)
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
train_data = dataset["train"]
test_data = dataset["test"]

# True labels for the test set (0: negative, 1: neutral, 2: positive)
y_true = test_data["label"]

# Extract sentences and guarantee they are standard Python strings
test_sentences = [str(sentence) for sentence in test_data["sentence"]]

# Function to compute metrics
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro")
    }

# 2. Evaluate Baseline: Zero-Shot General Model (RoBERTa)
print("Evaluating Baseline (RoBERTa)...")
baseline_pipeline = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

def map_sst2_to_finbank(prediction):
    label = prediction['label']
    if label == "NEGATIVE": return 0
    if label == "POSITIVE": return 2
    return 1 # Neutral fallback

baseline_preds = []
for p in baseline_pipeline(test_sentences, batch_size=16, truncation=True):
    baseline_preds.append(map_sst2_to_finbank(p))

baseline_metrics = compute_metrics(y_true, baseline_preds)

# 3. Evaluate Upper Bound: Domain-Specific (FinBERT)
print("Evaluating Upper Bound (FinBERT)...")
finbert_pipeline = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    device=0 if torch.cuda.is_available() else -1
)

def map_finbert(prediction):
    label = prediction['label']
    if label == "negative": return 0
    if label == "neutral": return 1
    if label == "positive": return 2

finbert_preds = []
for p in finbert_pipeline(test_sentences, batch_size=16, truncation=True):
    finbert_preds.append(map_finbert(p))

finbert_metrics = compute_metrics(y_true, finbert_preds)

# 4. Fine-Tune our own Model (DistilBERT)
print("Fine-tuning DistilBERT on financial data...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Ensure inputs are cast to string during tokenization as well
    return tokenizer([str(s) for s in examples["sentence"]], padding="max_length", truncation=True)

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_test = test_data.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Updated to use modern `eval_strategy` parameter
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

def compute_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return compute_metrics(labels, predictions)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_trainer_metrics,
)

trainer.train()
finetuned_predictions = np.argmax(trainer.predict(tokenized_test).predictions, axis=-1)
finetuned_metrics = compute_metrics(y_true, finetuned_predictions)

# 5. Generate and Export High-Res Confusion Matrices (>150 PPI)
def plot_cm(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    # 300 DPI safely exceeds the 150 PPI poster guideline requirement
    plt.figure(figsize=(6,5), dpi=300)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg', 'Neu', 'Pos'], yticklabels=['Neg', 'Neu', 'Pos'])
    plt.title(title, fontsize=14, pad=10)
    plt.ylabel('Actual', fontsize=12)
    plt.xlabel('Predicted', fontsize=12)
    plt.savefig(filename, bbox_inches='tight')
    plt.close()

plot_cm(y_true, baseline_preds, "Baseline (General Sentiment) CM", "baseline_cm.png")
plot_cm(y_true, finbert_preds, "FinBERT (Domain-Pretrained) CM", "finbert_cm.png")
plot_cm(y_true, finetuned_predictions, "Fine-Tuned DistilBERT CM", "finetuned_cm.png")

# 6. Generate and Export High-Res Metrics Bar Chart (>150 PPI)
def plot_metrics_bar_chart(baseline_m, finbert_m, finetuned_m, filename="model_comparison_barchart.png"):
    models = ['Baseline', 'FinBERT', 'Fine-Tuned (Ours)']
    accuracy = [baseline_m['accuracy'], finbert_m['accuracy'], finetuned_m['accuracy']]
    macro_f1 = [baseline_m['macro_f1'], finbert_m['macro_f1'], finetuned_m['macro_f1']]

    x = np.arange(len(models))
    width = 0.35

    # 300 DPI safely exceeds the 150 PPI poster guideline requirement
    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
    rects1 = ax.bar(x - width/2, accuracy, width, label='Accuracy', color='#4C72B0')
    rects2 = ax.bar(x + width/2, macro_f1, width, label='Macro F1', color='#DD8452')

    # Text sizing is intentionally scaled up to ensure legibility when printed on A1 dimensions[cite: 2]
    ax.set_ylabel('Score', fontsize=14)
    ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=12)
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=12, loc='upper left')

    ax.bar_label(rects1, fmt='%.3f', padding=3, fontsize=11)
    ax.bar_label(rects2, fmt='%.3f', padding=3, fontsize=11)

    plt.savefig(filename, bbox_inches='tight')
    plt.close()

plot_metrics_bar_chart(baseline_metrics, finbert_metrics, finetuned_metrics)

print("Pipeline complete. Metrics saved.")
print(f"Baseline: {baseline_metrics}\nFinBERT: {finbert_metrics}\nFine-Tuned: {finetuned_metrics}")

Loading dataset...


README.md:   0%|          | 0.00/9.29k [00:00<?, ?B/s]

sentences_75agree/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B /  267kB            

sentences_75agree/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3453 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.241787,0.914616,0.892033


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.241787,0.914616,0.892033
2,No log,0.283690,0.905933,0.887685
3,0.268677,0.251067,0.921852,0.902788


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Pipeline complete. Metrics saved.
Baseline: {'accuracy': 0.2575976845151954, 'macro_f1': 0.25141714703012674}
FinBERT: {'accuracy': 0.9493487698986975, 'macro_f1': 0.9385191535232308}
Fine-Tuned: {'accuracy': 0.9218523878437048, 'macro_f1': 0.9027882593088808}
